In [1]:
# Cell 1: setup
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

MODEL = "claude-haiku-5-5"  # check docs.claude.com for current model IDs
VALID = ["Clinical", "Physiological", "Social", "Historical record", "Unclear"]

GOLDEN = Path("evals/conditions_golden_set.csv")
PROMPTS = Path("prompts")
RESULTS = Path("evals/results")
RESULTS.mkdir(parents=True, exist_ok=True)

In [9]:
# Cell 2: helpers (the course versions, trimmed)
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

THINKING = "disabled"   # change to "adaptive" to compare

def chat(messages, system=None, max_tokens=500):
    params = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": messages,
        "thinking": {"type": THINKING}
    }
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    if message.stop_reason == "max_tokens":
        print("Warning: output hit max_tokens, so the answer may be cut off")
    return "".join(block.text for block in message.content if block.type == "text")

In [3]:
# Cell 3: data and prompt loading
def load_dataset(split):
    df = pd.read_csv(GOLDEN, dtype=str).fillna("")
    return df[df["split"] == split].to_dict("records")


def load_prompt(version):
    return (PROMPTS / f"{version}.txt").read_text()

In [4]:
# Cell 4: run one prompt, parse, grade
def run_prompt(test_case, template):
    # .replace instead of .format, because the template contains JSON braces
    prompt = template.replace("{name}", test_case["description"])
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


def parse_output(text):
    # tolerant parser: grab the first {...} block, so no prefill is needed
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return None


def grade_by_code(parsed, expected):
    if parsed is None or parsed.get("category") not in VALID:
        return {"valid": False, "predicted": "INVALID", "score": 0}
    predicted = parsed["category"]
    return {"valid": True, "predicted": predicted, "score": int(predicted == expected)}


def run_test_case(test_case, template):
    output = run_prompt(test_case, template)
    parsed = parse_output(output)
    grade = grade_by_code(parsed, test_case["expected_category"])
    return {
        "code": test_case["code"],
        "description": test_case["description"],
        "expected": test_case["expected_category"],
        "predicted": grade["predicted"],
        "valid": grade["valid"],
        "score": grade["score"],
        "reasoning": parsed.get("reasoning", "") if parsed else "",
        "raw_output": output,
        "note": test_case["note"],
    }

In [ ]:
# Cell 5: report, save, run
def report(results, version, split):
    df = pd.DataFrame(results)
    invalid = int((~df["valid"]).sum())
    unclear = int((df["predicted"] == "Unclear").sum())
    critical = int(
        ((df["expected"] == "Clinical") & df["predicted"].isin(["Social", "Historical record"])).sum()
    )
    leaks = int(
        (df["expected"].isin(["Social", "Historical record"]) & df["predicted"].isin(["Clinical", "Physiological"])).sum()
    )
    print(f"{version} | {MODEL} | {split} | n={len(df)}")
    print(f"Accuracy {df['score'].mean():.1%} | invalid {invalid} | unclear {unclear} | critical errors {critical} | anchor leaks {leaks}\n")
    print(pd.crosstab(df["expected"], df["predicted"]), "\n")
    for label in ["Clinical", "Physiological", "Social", "Historical record"]:
        tp = ((df["expected"] == label) & (df["predicted"] == label)).sum()
        precision = tp / max((df["predicted"] == label).sum(), 1) if (df["predicted"] == label).any() else None
        recall = tp / (df["expected"] == label).sum() if (df["expected"] == label).any() else None
        print(f"{label}: precision {'n/a' if precision is None else f'{precision:.0%}'}, "
            f"recall {'n/a' if recall is None else f'{recall:.0%}'}")
    print("\nMisses:")
    print(df[df["score"] == 0][["code", "description", "expected", "predicted", "reasoning"]].to_string())


def save(results, version, split):
    df = pd.DataFrame(results)
    (RESULTS / f"{version}_{MODEL}_{THINKING}_{split}.json").write_text(json.dumps(results, indent=2))
    row = {
        "time": datetime.now().isoformat(timespec="seconds"),
        "version": version,
        "model": MODEL,
        "thinking": THINKING,
        "split": split,
        "n": len(df),
        "accuracy": round(df["score"].mean(), 3),
        "invalid": int((~df["valid"]).sum()),
        "unclear": int((df["predicted"] == "Unclear").sum()),
    }
    log = RESULTS / "runs.csv"
    pd.DataFrame([row]).to_csv(log, mode="a", header=not log.exists(), index=False)


def run_eval(split, version, example_codes=()):
    template = load_prompt(version)
    dataset = [t for t in load_dataset(split) if t["code"] not in set(example_codes)]
    results = [run_test_case(t, template) for t in dataset]
    report(results, version, split)
    save(results, version, split)
    return results

In [8]:
# Cell 6: baseline
results = run_eval("dev", "v0")

v0 | claude-haiku-5-5 | dev | n=100
Accuracy 97.0% | invalid 0 | unclear 0 | critical errors 0

predicted          Clinical  Historical record  Physiological  Social
expected                                                             
Clinical                 81                  0              0       0
Historical record         2                  1              0       0
Physiological             0                  0              1       0
Social                    1                  0              0      14 

Clinical: precision 96%, recall 100%
Physiological: precision 100%, recall 100%
Social: precision 100%, recall 93%
Historical record: precision 100%, recall 33%

Misses:
         code                                   description           expected predicted                                                                                                                                                                                                                                

In [11]:
EXAMPLES = {"72892002", "25064002", "19169002", "162864005", "73595000",
            "162573006", "160904001", "10939881000119104", "161622006"}

results_v0  = run_eval("dev", "v0", example_codes=EXAMPLES)
results_v0b = run_eval("dev", "v0", example_codes=EXAMPLES)  # noise check
results_v1  = run_eval("dev", "v1", example_codes=EXAMPLES)

v0 | claude-haiku-5-5 | dev | n=91
Accuracy 94.5% | invalid 0 | unclear 0 | critical errors 0 | anchor leaks 4

predicted          Clinical  Historical record  Social
expected                                              
Clinical                 76                  0       0
Historical record         2                  0       0
Social                    2                  1      10 

Clinical: precision 95%, recall 100%
Physiological: precision 0%, recall 0%
Social: precision 100%, recall 77%
Historical record: precision 0%, recall 0%

Misses:
             code                                                description           expected          predicted                                                                                                                                                                                                                                                    reasoning
73      361055000                                    Misuses drugs (finding)    